In [1]:
!pip install phe

  Using cached phe-1.5.0-py2.py3-none-any.whl.metadata (3.8 kB)
Using cached phe-1.5.0-py2.py3-none-any.whl (53 kB)


In [ ]:
!pip install haversine

In [5]:
import math
from phe import paillier
from haversine import haversine


#step 1: Convert degrees to radians and compute trigonometric values
def trigonometric_values(lat_A, lon_A, lat_B, lon_B):
# Convert degrees to radians
    latA = math.radians(lat_A)
    lonA = math.radians(lon_A)
    latB = math.radians(lat_B)
    lonB = math.radians(lon_B)

# Compute the trigonometric values as per the protocol
    alpha = math.cos(latA / 2)
    beta = math.sin(latB / 2)
    gamma = math.sin(latA / 2)
    delta = math.cos(latB / 2)
    zeta = math.cos(latA)
    eta = math.cos(latB)
    theta = math.sin(lonA / 2)
    lambda_ = math.cos(lonB / 2)
    mu = math.cos(lonA / 2)
    nu = math.sin(lonB / 2)
    return (alpha, beta, gamma, delta, zeta, eta, theta, lambda_, mu, nu)
    
# Step 2: Alice computes encrypted values with her public key and sends them to ther server.
def alice_encrypt_values(public_key, alpha, gamma, zeta, eta, theta, lambda_, mu):
    # Compute
    alpha_squared = alpha**2
    neg_two_alpha_gamma = -2 * alpha * gamma
    gamma_squared = gamma**2
    zeta_eta_theta_lambda_squared = zeta * eta * (theta**2) * (lambda_**2)
    neg_two_zeta_eta_theta_lambda = -2 * zeta * eta * theta * lambda_
    zeta_mu = zeta * mu**2

    # Encrypt the values
    enc_alpha_squared = public_key.encrypt(alpha_squared)
    enc_neg_two_alpha_gamma = public_key.encrypt(neg_two_alpha_gamma)
    enc_gamma_squared = public_key.encrypt(gamma_squared)
    enc_zeta_eta_theta_lambda_squared = public_key.encrypt(zeta_eta_theta_lambda_squared)
    enc_neg_two_zeta_eta_theta_lambda = public_key.encrypt(neg_two_zeta_eta_theta_lambda)
    enc_zeta_mu = public_key.encrypt(zeta_mu)

# Alice sends encrypted values to the server
    return {
        "enc_alpha_squared": enc_alpha_squared,
        "enc_neg_two_alpha_gamma": enc_neg_two_alpha_gamma,
        "enc_gamma_squared": enc_gamma_squared,
        "enc_zeta_eta_theta_lambda_squared": enc_zeta_eta_theta_lambda_squared,
        "enc_neg_two_zeta_eta_theta_lambda": enc_neg_two_zeta_eta_theta_lambda,
        "enc_zeta_eta": enc_zeta_mu,
    }

# Step 3: The server computes alice encrypted values using homomorphic operations and send them to Bob
def Server_compute_homomorphic_ecnryption(alice_data, beta, delta, mu, nu, eta):
    beta_squared = beta**2
    delta_squared = delta**2
    mu_nu = mu * nu
    eta_nu_squared = eta * nu**2

    enc_a = (
        alice_data["enc_alpha_squared"] * beta_squared
        + alice_data["enc_neg_two_alpha_gamma"] * (beta * delta)
        + alice_data["enc_gamma_squared"] * delta_squared
        + alice_data["enc_zeta_eta_theta_lambda_squared"]
        + alice_data["enc_neg_two_zeta_eta_theta_lambda"] * mu_nu
        + alice_data["enc_zeta_eta"] * eta_nu_squared
    )

    return enc_a

# Step 4: Bob decrypts enc_a with Alice private key and computes the distance
def Bob_compute_distance(enc_a,private_key, geofence_radius):
    a = private_key.decrypt(enc_a)
    R = 6371.0 # Earth's radius in kilometers
    distance = 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a)) # Compute haversine distance

    # Determine if Alice's point is inside or outside Bob geofence
    if distance <= geofence_radius:
        print(f"ALice's location is INSIDE the geofence (distance: {distance:.2f} km).")
    else:
        print(f"Alice's location is OUTSIDE the geofence (distance: {distance:.2f} km).")
        
    return distance

def main():
    public_key, private_key = paillier.generate_paillier_keypair()
    lat_A, lon_A = 50.379320, -4.131244  # Alice's coordinates
    lat_B, lon_B = 50.381813, -4.127100  # Bob's coordinates
    geofence_radius = 0.39  # Radius in kilometers

    # Step 1: Convert degrees to radians and compute trigonometric values
    alpha, beta, gamma, delta, zeta, eta, theta, lambda_, mu, nu = trigonometric_values(lat_A, lon_A, lat_B, lon_B)

    # Step 2: Alice computes encrypted values and sends them to the server
    alice_data = alice_encrypt_values(public_key, alpha, gamma, zeta, eta, theta, lambda_, mu)

    # Step 3: The server computes alice encrypted values using homomorphic operations
    enc_a = Server_compute_homomorphic_ecnryption(alice_data, beta, delta, mu, nu, eta)

    # Step 4: Bob decrypts enc_a and computes the distance
    distance = Bob_compute_distance(enc_a, private_key, geofence_radius)

    # Validate using standard haversine library to prove validaty
    haversine_distance = haversine((lat_A, lon_A), (lat_B, lon_B))
    print(f"Distance between Alice and Bob (using haversine library): {haversine_distance:.2f} km")

if __name__ == "__main__":
    main()

Alice's location is OUTSIDE the geofence (distance: 0.40 km).
Distance between Alice and Bob (using haversine library): 0.40 km


In [ ]:
import math
import time
import csv
from phe import paillier
from haversine import haversine

# Step 1: Convert degrees to radians and compute trigonometric values
def trigonometric_values(lat_A, lon_A, lat_B, lon_B):
    latA = math.radians(lat_A)
    lonA = math.radians(lon_A)
    latB = math.radians(lat_B)
    lonB = math.radians(lon_B)

    alpha = math.cos(latA / 2)
    beta = math.sin(latB / 2)
    gamma = math.sin(latA / 2)
    delta = math.cos(latB / 2)
    zeta = math.cos(latA)
    eta = math.cos(latB)
    theta = math.sin(lonA / 2)
    lambda_ = math.cos(lonB / 2)
    mu = math.cos(lonA / 2)
    nu = math.sin(lonB / 2)
    return (alpha, beta, gamma, delta, zeta, eta, theta, lambda_, mu, nu)

# Step 2: Alice computes encrypted values with her public key and sends them to the server.
def alice_encrypt_values(public_key, alpha, gamma, zeta, eta, theta, lambda_, mu):
    alpha_squared = alpha**2
    neg_two_alpha_gamma = -2 * alpha * gamma
    gamma_squared = gamma**2
    zeta_eta_theta_lambda_squared = zeta * eta * (theta**2) * (lambda_**2)
    neg_two_zeta_eta_theta_lambda = -2 * zeta * eta * theta * lambda_
    zeta_mu = zeta * mu**2

    enc_alpha_squared = public_key.encrypt(alpha_squared)
    enc_neg_two_alpha_gamma = public_key.encrypt(neg_two_alpha_gamma)
    enc_gamma_squared = public_key.encrypt(gamma_squared)
    enc_zeta_eta_theta_lambda_squared = public_key.encrypt(zeta_eta_theta_lambda_squared)
    enc_neg_two_zeta_eta_theta_lambda = public_key.encrypt(neg_two_zeta_eta_theta_lambda)
    enc_zeta_mu = public_key.encrypt(zeta_mu)

    return {
        "enc_alpha_squared": enc_alpha_squared,
        "enc_neg_two_alpha_gamma": enc_neg_two_alpha_gamma,
        "enc_gamma_squared": enc_gamma_squared,
        "enc_zeta_eta_theta_lambda_squared": enc_zeta_eta_theta_lambda_squared,
        "enc_neg_two_zeta_eta_theta_lambda": enc_neg_two_zeta_eta_theta_lambda,
        "enc_zeta_eta": enc_zeta_mu,
    }

# Step 3: The server computes Alice's encrypted values using homomorphic operations and sends them to Bob
def Server_compute_homomorphic_encryption(alice_data, beta, delta, mu, nu, eta):
    beta_squared = beta**2
    delta_squared = delta**2
    mu_nu = mu * nu
    eta_nu_squared = eta * nu**2

    enc_a = (
        alice_data["enc_alpha_squared"] * beta_squared
        + alice_data["enc_neg_two_alpha_gamma"] * (beta * delta)
        + alice_data["enc_gamma_squared"] * delta_squared
        + alice_data["enc_zeta_eta_theta_lambda_squared"]
        + alice_data["enc_neg_two_zeta_eta_theta_lambda"] * mu_nu
        + alice_data["enc_zeta_eta"] * eta_nu_squared
    )

    return enc_a

# Step 4: Bob decrypts enc_a with Alice's private key and computes the distance
def Bob_compute_distance(enc_a, private_key, geofence_radius):
    a = private_key.decrypt(enc_a)
    R = 6371.0  # Earth's radius in kilometers
    distance = 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))  # Compute haversine distance

    if distance <= geofence_radius:
        return True  # Inside geofence
    else:
        return False  # Outside geofence

def main():
    public_key, private_key = paillier.generate_paillier_keypair()
    lat_A, lon_A = 50.379320, -4.131244  # Alice's coordinates
    lat_B, lon_B = 50.381813, -4.127100  # Bob's coordinates
    geofence_radius = 0.39  # Radius in kilometers

    # Store timing results
    results = []

    for _ in range(30):
        # Measure time for trigonometric values
        start_time = time.time()
        alpha, beta, gamma, delta, zeta, eta, theta, lambda_, mu, nu = trigonometric_values(lat_A, lon_A, lat_B, lon_B)
        trigonometric_time = time.time() - start_time

        # Measure time for encryption
        start_time = time.time()
        alice_data = alice_encrypt_values(public_key, alpha, gamma, zeta, eta, theta, lambda_, mu)
        encryption_time = time.time() - start_time

        # Measure time for homomorphic encryption
        start_time = time.time()
        enc_a = Server_compute_homomorphic_encryption(alice_data, beta, delta, mu, nu, eta)
        calculation_time = time.time() - start_time

        # Measure time for decryption and distance calculation
        start_time = time.time()
        distance_result = Bob_compute_distance(enc_a, private_key, geofence_radius)
        decryption_time = time.time() - start_time

        # Append results
        results.append((trigonometric_time, encryption_time, calculation_time, decryption_time))

    # Calculate averages
    avg_times = [sum(x) / len(x) for x in zip(*results)]

    # Save results to CSV
    with open('timing_results.csv', mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Step", "Average Time (seconds)"])
        writer.writerow(["Trigonometric Values", avg_times[0]])
        writer.writerow(["Encryption", avg_times[1]])
        writer.writerow(["Calculation", avg_times[2]])
        writer.writerow(["Decryption", avg_times[3]])

    print("Average times.csv")

if __name__ == "__main__":
    main()


Average times saved to timing_results.csv


In [3]:
import pandas as pd
import math
from phe import paillier
from haversine import haversine

# Load the dataset
data = pd.read_csv("places.csv") 

# User input for geofence parameters
center_lat = float(input("Enter center latitude: "))
center_lon = float(input("Enter center longitude: "))
geofence_radius = float(input("Enter geofence radius (km): "))

# Step 1: Key Generation 
public_key, private_key = paillier.generate_paillier_keypair()

# Define a function for privacy-preserving haversine distance
# make three functionaliteis
#user send cyphertext to the server
# server do the calculation
#client decrpyt the distance cyper text recieved from the server and compare the geofecne-radius to check iof the user outside or inside
#the geofence
#test each fucntion for the fucntionalities
def is_within_geofence(lat_A, lon_A, geofence_radius, lat_B, lon_B):
    # Convert degrees to radians
    latA = math.radians(lat_A)
    lonA = math.radians(lon_A)
    latB = math.radians(lat_B)
    lonB = math.radians(lon_B)

    # Compute trigonometric values
    alpha = math.cos(latA / 2)
    beta = math.sin(latB / 2)
    gamma = math.sin(latA / 2)
    delta = math.cos(latB / 2)
    zeta = math.cos(latA)
    eta = math.cos(latB)
    theta = math.sin(lonA / 2)
    lambda_ = math.cos(lonB / 2)
    mu = math.cos(lonA / 2)
    nu = math.sin(lonB / 2)

    # Computes encrypted values
    enc_alpha_squared = public_key.encrypt(alpha**2)
    enc_neg_two_alpha_gamma = public_key.encrypt(-2 * alpha * gamma)
    enc_gamma_squared = public_key.encrypt(gamma**2)
    enc_zeta_eta_theta_lambda_squared = public_key.encrypt(zeta * eta * (theta**2) * (lambda_**2))
    enc_neg_two_zeta_eta_theta_lambda = public_key.encrypt(-2 * zeta * eta * theta * lambda_)
    enc_zeta_mu = public_key.encrypt(zeta * mu**2)

    # Computes encrypted distance
    beta_squared = beta**2
    delta_squared = delta**2
    mu_nu = mu * nu
    eta_nu_squared = eta * nu**2

    enc_a = (
        enc_alpha_squared * beta_squared +
        enc_neg_two_alpha_gamma * (beta * delta) +
        enc_gamma_squared * delta_squared +
        enc_zeta_eta_theta_lambda_squared +
        enc_neg_two_zeta_eta_theta_lambda * mu_nu +
        enc_zeta_mu * eta_nu_squared
    )

    # Decrypts a
    a = private_key.decrypt(enc_a)

    # Compute haversine distance
    R = 6371.0  # Earth's radius in km
    distance = 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return distance <= geofence_radius, distance

# Iterate through the dataset
for index, row in data.iterrows():
    point_lat = row['latitude']
    point_lon = row['longitude']

    inside, dist = is_within_geofence(center_lat, center_lon, geofence_radius, point_lat, point_lon)

    # Validate using standard haversine library
    haversine_distance = haversine((center_lat, center_lon), (point_lat, point_lon))
    print(f"Standard Haversine distance: {haversine_distance:.2f} km")

    if inside:
        print(f"{row['name']} is INSIDE the geofence (distance: {dist:.2f} km).")
    else:
        print(f"{row['name']} is OUTSIDE the geofence (distance: {dist:.2f} km).")


ValueError: could not convert string to float: ''